# Week 34, Day 4: DRY, KISS and YAGNI

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w034d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def basket_vat(pence):
    return pence * 20 // 100

def receipt_vat(pence):
    return pence * 17 // 100

def invoice_vat(pence):
    return pence * 20 // 100

print(basket_vat(1000), receipt_vat(1000), invoice_vat(1000))

In [ ]:
VAT_RATE = 20                              # 1. a named constant for a number that means something

def vat(pence: int) -> int:                # 2. a function for a rule used in several places
    return pence * VAT_RATE // 100

GRADES = [(70, "A"), (60, "B"), (50, "C"), (40, "D")]     # 3. a table for data written as code

def grade(mark: int) -> str:
    for lowest, letter in GRADES:
        if mark >= lowest:
            return letter
    return "U"

print(vat(1000), vat(999), [grade(m) for m in [85, 60, 59, 12]])

In [ ]:
def max3_clever(a, b, c):
    return a if a > b and a > c else b if b > c else c

def max3_simple(a, b, c):
    return max(a, b, c)

def has_overdue_clever(loans, today):
    return bool(len([1 for l in loans if l[1] < today]) > 0)

def has_overdue_simple(loans, today):
    return any(due < today for _, due in loans)

print(max3_clever(5, 5, 1), max3_simple(5, 5, 1))
print(has_overdue_simple([("Aké", 3)], 5), has_overdue_clever([("Aké", 3)], 5))

> **Key idea.** DRY (Hunt and Thomas): every piece of knowledge has one authoritative place; use named constants, helper functions and tables of data. Code that merely looks alike may represent different decisions, so merge only shared knowledge; follow the rule of three, and prefer duplication to the wrong abstraction (Sandi Metz). KISS: the simplest clear code is best; use the standard library, return early, name things. YAGNI (Extreme Programming): don't build options, layers or extension points before they're needed; keep the code easy to change instead. Balance them with judgement.

### Exercise: One home for the VAT rate

The VAT rule is written three times. Introduce a constant `VAT_RATE = 20` and a function `vat(pence)` (rounded down), and use `vat` in all three functions. The checks change `VAT_RATE` and expect every function to follow.

In [ ]:
def basket_total(prices):
    net = sum(prices)
    return net + net * 20 // 100

def receipt_line(name, pence):
    return f"{name}: {pence} + {pence * 20 // 100} VAT"

def invoice_vat(lines):
    return sum(qty * pence * 20 // 100 for qty, pence in lines)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_one_place():\n    \"a single source of truth\"\n    assert basket_total([500, 500]) == 1200 and receipt_line(\"Rice\", 450) == \"Rice: 450 + 90 VAT\" and invoice_vat([(2, 500), (1, 999)]) == 399\n    ns[\"VAT_RATE\"] = 5\n    try:\n        assert vat(1000) == 50 and basket_total([1000]) == 1050 and receipt_line(\"Oil\", 1000) == \"Oil: 1000 + 50 VAT\"\n        assert invoice_vat([(1, 1000)]) == 50, \"every function should use the one rate\"\n    finally:\n        ns[\"VAT_RATE\"] = 20\n    assert source.count(\"20\") == 1, \"the number 20 should appear once\"")

### Exercise: A table instead of a chain

Replace the `if`/`elif` chain with a table `BANDS`, a list of `(upper_limit, price)` pairs, and a short loop. Parcels up to and including 1000 g cost 350, up to 2000 g cost 500, up to 5000 g cost 800, up to 10000 g cost 1200; anything heavier raises `ValueError`, as does a weight of 0 or less.

In [ ]:
def postage(grams):
    if grams <= 0:
        raise ValueError("weight must be positive")
    if grams <= 1000:
        return 350
    elif grams <= 2000:
        return 500
    elif grams <= 5000:
        return 800
    elif grams <= 10000:
        return 1200
    raise ValueError("too heavy")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_table():\n    \"table-driven\"\n    assert [postage(g) for g in [1, 1000, 1001, 2000, 2001, 5000, 10000]] == [350, 350, 500, 500, 800, 800, 1200]\n    for bad in [0, -5, 10001]:\n        try:\n            postage(bad)\n            assert False, f\"postage({bad}) should raise ValueError\"\n        except ValueError:\n            pass\n    assert \"elif\" not in source and isinstance(ns.get(\"BANDS\"), list)\n    ns[\"BANDS\"].append((20000, 2000))\n    try:\n        assert postage(15000) == 2000, \"adding a band should only need a new entry\"\n    finally:\n        ns[\"BANDS\"].pop()")

### Exercise: Simpler, same behaviour

`summary` works but is far more complicated than it needs to be. Rewrite it to do exactly the same, using built-ins (`sum`, `max`, `min`, `any`, `sorted`) so that its body is **no more than 6 lines**. It returns `None` for an empty list.

In [ ]:
def summary(marks):
    if len(marks) == 0:
        result = None
    else:
        total = 0
        for i in range(len(marks)):
            total = total + marks[i]
        highest = marks[0]
        for m in marks:
            if m > highest:
                highest = m
        failed = False
        for m in marks:
            if m < 40:
                failed = True
                break
        ordered = []
        for m in marks:
            ordered.append(m)
        ordered.sort()
        result = {"total": total, "highest": highest, "any_failed": failed, "ordered": ordered}
    return result

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_simple():\n    \"same behaviour, fewer lines\"\n    import ast\n    xs = [70, 35, 88]\n    assert summary(xs) == {\"total\": 193, \"highest\": 88, \"any_failed\": True, \"ordered\": [35, 70, 88]} and xs == [70, 35, 88]\n    assert summary([50]) == {\"total\": 50, \"highest\": 50, \"any_failed\": False, \"ordered\": [50]} and summary([]) is None\n    fn = next(n for n in ast.walk(ast.parse(source)) if isinstance(n, ast.FunctionDef) and n.name == \"summary\")\n    lines = fn.end_lineno - fn.body[0].lineno + 1\n    assert lines <= 6, f\"the body is {lines} lines; aim for 6 or fewer\"")

### Exercise: Options nobody uses

Every call to `send_invite` in the program passes only a name and an email; the other parameters were added "in case". Remove the unused parameters and the code that only they needed, keeping exactly what the calls below produce.

In [ ]:
def send_invite(name, email, language="en", template=None, cc=None, retries=3, dry_run=False):
    if template is None:
        template = "Hi {name}, you're invited!"
    if language == "fr":
        template = "Bonjour {name}, vous êtes invité !"
    message = template.format(name=name)
    recipients = [email] + (cc or [])
    for attempt in range(retries):
        if dry_run:
            return ("DRY RUN", recipients, message)
        return (email, message)

print(send_invite("Ada", "ada@example.org"))
print(send_invite("Bo", "bo@example.org"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lean():\n    \"no speculative options\"\n    import inspect\n    assert list(inspect.signature(send_invite).parameters) == [\"name\", \"email\"]\n    expect_output(\"('ada@example.org', \\\"Hi Ada, you're invited!\\\")\\n('bo@example.org', \\\"Hi Bo, you're invited!\\\")\")\n    for word in [\"retries\", \"dry_run\", \"language\", \"template\", \"cc\"]:\n        assert word not in source, f\"remove {word}\"")

### Exercise: Undoing the wrong abstraction

`label` started as a shared helper for shelf labels and web pages, then grew flags for each. Replace it with two plain functions, `shelf_label(name, pence)` (like `RICE £4.50`) and `web_label(name, pence, offer=False)` (like `<b>Rice</b> £4.50`, with ` (offer!)` added when `offer` is true), and delete `label`. Keep one small helper `money(pence)` for the shared part that really is the same knowledge.

In [ ]:
def label(name, pence, for_web=False, upper=True, bold=False, offer=False):
    text = name.upper() if upper and not for_web else name
    if bold and for_web:
        text = f"<b>{text}</b>"
    price = f"£{pence // 100}.{pence % 100:02}"
    result = f"{text} {price}"
    if offer and for_web:
        result += " (offer!)"
    return result

def shelf(name, pence):
    return label(name, pence)

def web(name, pence, offer=False):
    return label(name, pence, for_web=True, upper=False, bold=True, offer=offer)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_split():\n    \"two plain functions\"\n    import inspect\n    assert shelf_label(\"Rice\", 450) == \"RICE \u00a34.50\" and web_label(\"Rice\", 450) == \"<b>Rice</b> \u00a34.50\"\n    assert web_label(\"Oil\", 905, offer=True) == \"<b>Oil</b> \u00a39.05 (offer!)\" and money(7) == \"\u00a30.07\"\n    assert \"label\" not in ns and list(inspect.signature(shelf_label).parameters) == [\"name\", \"pence\"]\n    assert source.count(\"// 100\") == 1, \"keep the money format in one place\"")

### Bug hunt: Two rates for one rule

The basket and the receipt disagree about the delivery charge for the same order, because the rule is written twice and one copy is out of date. Free delivery starts at £40 (4000 pence); below that it costs £3.99. Fix it by putting the rule in **one** function, `delivery(pence)`, used by both, so the program prints `4398 4398` then `4000 4000`.

In [ ]:
def basket_total(pence):
    return pence + (0 if pence >= 4000 else 399)

def receipt_total(pence):
    return pence + (0 if pence >= 3000 else 399)

for amount in [3999, 4000]:
    print(basket_total(amount), receipt_total(amount))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one rule\"\n    expect_output(\"4398 4398\\n4000 4000\")\n    assert delivery(3999) == 399 and delivery(4000) == 0 and source.count(\">=\") == 1, \"write the rule once\"")

### Bug hunt: Too clever by half

`biggest` should return the largest of three numbers, but `biggest(5, 1, 9)` gives `5`. Replace the clever line with something simple and correct, so the program prints `[9, 9, 9, 7]`.

In [ ]:
def biggest(a, b, c):
    return a if a > b else b if b > c else c

print([biggest(5, 1, 9), biggest(9, 5, 1), biggest(1, 9, 5), biggest(7, 7, 7)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the largest\"\n    expect_output(\"[9, 9, 9, 7]\")\n    assert biggest(-1, -5, -3) == -1")

### Bug hunt: A copy-and-paste slip

The totals for two shops were written by copying a loop, and one variable wasn't renamed, so Yaba's total includes Ikeja's sales: it prints `Ikeja 2100` then `Yaba 3600`. Remove the duplication by writing one function `shop_total(sales, shop)`, used for both, so it prints `Ikeja 2100` then `Yaba 1500`.

In [ ]:
sales = [("Ikeja", 1500), ("Yaba", 1500), ("Ikeja", 600)]

ikeja = 0
for shop, pence in sales:
    if shop == "Ikeja":
        ikeja += pence

yaba = ikeja
for shop, pence in sales:
    if shop == "Yaba":
        yaba += pence

print("Ikeja", ikeja)
print("Yaba", yaba)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"one function\"\n    expect_output(\"Ikeja 2100\\nYaba 1500\")\n    assert shop_total([(\"A\", 1), (\"B\", 2), (\"A\", 3)], \"A\") == 4 and source.count(\"for \") == 1")

### Puzzle: A table of rules

Arrange the lines into a table-driven function for a cinema's ticket prices by age. One line is a trap.

(Put these lines in order on the website.)

### Exercise: DRY, KISS or YAGNI?

For each situation, which principle is being broken? Answer `"DRY"`, `"KISS"` or `"YAGNI"`.

In [ ]:
# p: The shop's opening hours are typed into five different functions.
# q: A function has a plugin parameter, but the program has only ever had one plugin.
# r: Counting words with a hand-written loop of fifteen lines instead of collections.Counter.
# s: The list of valid countries is in the code and again in a settings file.
# t: A base class and a factory were written so that a second kind of report "could be added one day".

p = ___
q = ___
r = ___
s = ___
t = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (\"DRY\", \"YAGNI\", \"KISS\"), \"p to r\"\n    assert (s, t) == (\"DRY\", \"YAGNI\")")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: one table for a whole report

`report` builds a table of products with three columns, and the knowledge of each column (its heading, its width, how to get its value) is spread over three separate places. Rewrite it so each column is described **once**, in a list `COLUMNS` of `Column` objects (a frozen dataclass with `heading`, `width` and `get`, a function from a product dictionary to the text to show), and `report(products)` builds the header, a line of dashes and one row per product entirely from `COLUMNS`. Each cell is left-aligned to its column's width and cells are separated by a single space, with trailing spaces removed from every line. The output must be exactly the same as before. The checks add a column to `COLUMNS` and expect it to appear with no other change.

In [ ]:
from dataclasses import dataclass
from collections.abc import Callable

def report(products):
    lines = []
    lines.append(f"{'Name':<10} {'Price':<8} {'Stock':<5}".rstrip())
    lines.append(f"{'-' * 10} {'-' * 8} {'-' * 5}")
    for p in products:
        price = f"£{p['pence'] // 100}.{p['pence'] % 100:02}"
        lines.append(f"{p['name']:<10} {price:<8} {str(p['stock']):<5}".rstrip())
    return "\n".join(lines)

PRODUCTS = [{"name": "Rice", "pence": 450, "stock": 12}, {"name": "Palm oil", "pence": 900, "stock": 3}]
print(report(PRODUCTS))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PRODUCTS = [{\"name\": \"Rice\", \"pence\": 450, \"stock\": 12}, {\"name\": \"Palm oil\", \"pence\": 900, \"stock\": 3}]\ndef test_same():\n    \"the same report\"\n    want = \"Name       Price    Stock\\n---------- -------- -----\\nRice       \u00a34.50    12\\nPalm oil   \u00a39.00    3\"\n    assert report(PRODUCTS) == want, \"got:\\n\" + report(PRODUCTS)\n    import dataclasses\n    C = ns[\"Column\"]\n    assert dataclasses.is_dataclass(C) and [f.name for f in dataclasses.fields(C)] == [\"heading\", \"width\", \"get\"] and len(COLUMNS) == 3\ndef test_extend():\n    \"a new column is one entry\"\n    C = ns[\"Column\"]\n    COLUMNS.append(C(\"Value\", 7, lambda p: str(p[\"pence\"] * p[\"stock\"])))\n    try:\n        got = report(PRODUCTS)\n        assert got == \"Name       Price    Stock Value\\n---------- -------- ----- -------\\nRice       \u00a34.50    12    5400\\nPalm oil   \u00a39.00    3     2700\", \"got:\\n\" + got\n    finally:\n        COLUMNS.pop()\n    assert report([]) == \"Name       Price    Stock\\n---------- -------- -----\"")

## Warm-up

Someone updated the VAT rate in two places and missed the third (or the rate was 17.5 per cent once, and this one was half-updated). The basket, the receipt and the invoice now disagree, and nothing crashes: customers just get receipts that don't add up. The rule "VAT is 20 per cent" was written three times, so it could change in two places and not the third. That's what today's first principle prevents.

## Don't repeat yourself

**DRY** comes from *The Pragmatic Programmer* by Andrew Hunt and David Thomas (1999), which puts it like this: "every piece of knowledge must have a single, unambiguous, authoritative representation within a system." The key word is **knowledge**: a rule, a number, a format, a decision. When it lives in one place, changing it is one edit, and the program can't disagree with itself.

Three everyday tools remove duplicated knowledge:

Compare the table with the usual `if mark >= 70: return "A"` `elif mark >= 60: ...`: the boundaries are now data that can be read, tested, printed in a help message, or even loaded from a file, and adding a grade is one line.

DRY applies beyond code: a value in a settings file and again in the code, a rule in the documentation and again in the program, a database column and a hand-copied list of the same names. Wherever the same fact is written twice, the two copies will eventually disagree.

## Duplication that isn't

Not every repetition is duplicated knowledge. Two pieces of code can look the same and still mean different things:

```python
def valid_username(name):
    return len(name) >= 8

def valid_password(password):
    return len(password) >= 8
```

These share text, but not knowledge: the username rule and the password rule are separate decisions that happen to agree today. Merging them into one `long_enough` function couples two rules that will change for different reasons (security advice changes; usernames don't), which is yesterday's single responsibility problem.

Two pieces of advice help:

- **The rule of three.** The first time, just write it. The second time, notice the repetition, but tolerate it. The third time, it's a real pattern: extract it. By then you know what actually varies. (The rule is credited to Don Roberts, and was made widely known by Martin Fowler's book *Refactoring*.)
- **The wrong abstraction is worse than duplication.** A shared function that grows a flag for each caller (`format(x, html=True, short=False, legacy=True)`) ties all its callers together. As Sandi Metz put it, "duplication is far cheaper than the wrong abstraction". If a shared helper keeps needing special cases, it's often better to copy it back into its callers and let each be simple.

## Keep it simple

**KISS** ("keep it simple") asks for the simplest design that does the job. Simple doesn't mean short, and it doesn't mean clever. It means **easy to read and hard to get wrong**:

Habits that keep code simple:

- **Use the standard library.** `max`, `sum`, `any`, `all`, `sorted`, `Counter`, `defaultdict` are tested, fast and instantly readable. Re-writing them by hand adds places for bugs.
- **Return early.** Deal with the special cases first and return, so the main path isn't buried in nested `if`s (Day 5 practises this).
- **Name things.** A well-named variable or helper function often replaces a comment and a puzzle.
- **Choose plain structures.** A dictionary or a dataclass before a class hierarchy; a function before a class with one method (Week 31); one module before a package of five.

Clever code has a real cost: whoever reads it next (often you, months later) must work out what it does before they can change it.

## You aren't going to need it

**YAGNI** comes from Extreme Programming (Ron Jeffries and others, late 1990s): **don't build something until you actually need it.** It targets a very common habit: adding options, layers and extension points "in case" a future requirement arrives.

```python
def export_report(rows, fmt="csv", compress=False, encrypt=False, legacy_mode=False,
                  plugin=None, retries=3, verbose=False):
    ...
```

If every caller in the program uses `export_report(rows)`, the other seven parameters are code that must be read, tested and kept working, for features nobody has. Worse, guesses about the future are usually wrong in the details, so when the real requirement arrives, the speculative code often doesn't fit it anyway and gets rewritten.

YAGNI isn't an excuse for poor design: keep code clean and easy to change (cohesive, loosely coupled, tested), so that when the need is real, adding it is cheap. That's what makes it safe not to build it early.

Signs of speculative code:

- parameters that no caller passes, or always passes the same value;
- an interface, registry or base class with exactly one implementation and no second in sight;
- configuration options nobody sets;
- "for later" functions that nothing calls (`git` remembers deleted code, so it isn't lost).

## Balancing them

These principles pull against each other and against SOLID, and judgement is the point:

| Tension | How to decide |
|---|---|
| DRY against single responsibility | merge only code that represents the **same knowledge** |
| open-closed against YAGNI | add an extension point when a second case is real or clearly coming, not "just in case" |
| KISS against everything | if a principle makes the code harder to understand, it's being overdone |

## Practice

## Quiz

## Daily challenge

## Before you go

Search a project of your own for a number or a word that appears in several places and means the same thing each time (a rate, a limit, a file name), and give it one home. Then look for a parameter that no caller uses. Tomorrow: refactoring step by step, changing a program's design safely while keeping what it does exactly the same.